In [2]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [3]:
from pkgimp import *
from bson import ObjectId
from tqdm import tqdm

from nb2p import database, fileop, config, astparse
from nb2p.notebook import Notebook

In [4]:
DATASET_NAME = "distilkaggle"

In [5]:
DIRS = config.dirs(dataset_name=DATASET_NAME)
DIRS.makedirs()

making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb


Connect to database.

In [6]:
db, client = database.connect(dataset_name=DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


## Get Test Notebooks

In [7]:
nb_ids = sorted(
    list(
        map(
            lambda x: x["notebook_id"],
            db.notebooksegments.find({"selected": True}, {"notebook_id": 1}),
        )
    ),
)
len(nb_ids), nb_ids[:3]

(1024, [19349, 23162, 28981])

In [8]:
RAW_DIR = DIRS.base / "raw"

In [9]:
code_df = pd.read_csv(RAW_DIR / "code.csv")
code_df

/tmp/ipykernel_3882357/753515395.py:1: DtypeWarning: Columns (0,1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  code_df = pd.read_csv(RAW_DIR / "code.csv")


,kernel_id,cell_index,source,output_type,execution_count
0,12034794,3,from mpl_toolkits.mplot3d import Axes3D\nfrom ...,NaN,1.0
1,12034794,5,"for dirname, _, filenames in os.walk('/kaggle/...",stream,2.0
2,12034794,7,# Distribution graphs (histogram/bar graph) of...,NaN,3.0
3,12034794,8,# Correlation matrix\ndef plotCorrelationMatri...,NaN,4.0
4,12034794,9,# Scatter and density plots\ndef plotScatterMa...,NaN,5.0
...,...,...,...,...,...
12195425,203109,7,# From plot above we can make desicion how man...,stream,6.0
12195426,203109,8,# From plot above we can see that roughly spea...,stream,7.0
12195427,203109,9,"# Roughly speaking, the message contains 10 wo...",execute_result,8.0
12195428,203109,10,"# At the end, let's show up how are messages d...",display_data,9.0


In [10]:
code_df = code_df.rename(columns={"kernel_id": "notebook_id", "cell_index": "cell_id"})
code_df["cell_type"] = "code"
code_df[:3]

,notebook_id,cell_id,source,output_type,execution_count,cell_type
0,12034794,3,from mpl_toolkits.mplot3d import Axes3D\nfrom ...,NaN,1.0,code
1,12034794,5,"for dirname, _, filenames in os.walk('/kaggle/...",stream,2.0,code
2,12034794,7,# Distribution graphs (histogram/bar graph) of...,NaN,3.0,code


In [11]:
selected_df = code_df.loc[code_df.notebook_id.isin(nb_ids)]
selected_df = selected_df.sort_values(["notebook_id", "cell_id"])
selected_df

,notebook_id,cell_id,source,output_type,execution_count,cell_type
11988467,19349,1.0,%matplotlib inline,NaN,NaN,code
11988468,19349,3.0,import pandas as pd\nimport sqlite3\ncon = sql...,NaN,NaN,code
11988469,19349,5.0,"print(pd.read_sql_query(""""""\nSELECT c.Competit...",NaN,NaN,code
11988470,19349,7.0,"top10 = pd.read_sql_query(""""""\nSELECT *\nFROM ...",NaN,NaN,code
11988471,19349,9.0,"print(pd.read_sql_query(""""""\nSELECT *\nFROM Us...",NaN,NaN,code
...,...,...,...,...,...,...
3715355,31804779,53,"data[data[""gender""]==""male""].groupby(""country""...",execute_result,28.0,code
3715356,31804779,55,"data[['username','age']][ (data['age'] > 10) &...",execute_result,29.0,code
3715357,31804779,57,"data[""postcode""][data[""country""]==""Mexico""]",execute_result,30.0,code
3715358,31804779,59,"data[""age""].value_counts().sort_values(ascendi...",execute_result,31.0,code


In [12]:
markdown_df = pd.read_csv(RAW_DIR / "markdown.csv")
markdown_df

/tmp/ipykernel_3882357/3224147727.py:1: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  markdown_df = pd.read_csv(RAW_DIR / "markdown.csv")


,kernel_id,cell_index,source
0,13807716,1,## Introduction\nGreetings from the Kaggle bot...
1,13807716,2,## Exploratory Analysis\nTo begin this explora...
2,13807716,4,There is 0 csv file in the current version of ...
3,13807716,6,The next hidden code cells define functions fo...
4,13807716,10,"Oh, no! There are no automatic insights availa..."
...,...,...,...
5471395,1699835.0,15,Distribution graphs (histogram/bar graph) of s...
5471396,1699835.0,17,Correlation matrix:
5471397,1699835.0,19,Scatter and density plots:
5471398,1699835.0,21,## Conclusion\r\nThis concludes your starter a...


In [13]:
markdown_df = markdown_df.rename(
    columns={"kernel_id": "notebook_id", "cell_index": "cell_id"}
)
markdown_df["cell_type"] = "markdown"
markdown_df[:3]

,notebook_id,cell_id,source,cell_type
0,13807716,1,## Introduction\nGreetings from the Kaggle bot...,markdown
1,13807716,2,## Exploratory Analysis\nTo begin this explora...,markdown
2,13807716,4,There is 0 csv file in the current version of ...,markdown


In [14]:
md_df = markdown_df.loc[markdown_df.notebook_id.isin(nb_ids)]
md_df = md_df.sort_values(["notebook_id", "cell_id"])
md_df

,notebook_id,cell_id,source,cell_type
5368454,19349.0,2,# Exploring with a Python Notebook\n\nThe data...,markdown
5368455,19349.0,4,Let's explore the top 10 competitions by the n...,markdown
5368456,19349.0,6,We can look at the top 10 ranked users as foll...,markdown
5368457,19349.0,8,"In a similar vein, we can see all the users wh...",markdown
5368458,19349.0,10,Notebooks are great for showing visualizations...,markdown
...,...,...,...,...
2226757,31804779,52,21. Mostrar en consola el país con más hombres.,markdown
2226758,31804779,54,"22. Mostrar en consola el nombre, username y e...",markdown
2226759,31804779,56,23. Mostrar en consola el código postal de tod...,markdown
2226760,31804779,58,24. Obtener la edad que más se repite en el Da...,markdown


In [15]:
selected_with_md_df = pd.concat([selected_df, md_df])
selected_with_md_df = selected_with_md_df.sort_values(["notebook_id", "cell_id"])

for c in ["notebook_id", "cell_id"]:
    selected_with_md_df[c] = selected_with_md_df[c].round().astype(int)

selected_with_md_df = selected_with_md_df[selected_with_md_df["source"].str.len() > 0]
selected_with_md_df["repository_id"] = selected_with_md_df["notebook_id"]
selected_with_md_df = selected_with_md_df.reset_index(drop=True)
selected_with_md_df = selected_with_md_df.reset_index()
selected_with_md_df = selected_with_md_df.drop(columns=["cell_id"])

selected_with_md_df

,index,notebook_id,source,output_type,execution_count,cell_type,repository_id
0,0,19349,%matplotlib inline,NaN,NaN,code,19349
1,1,19349,# Exploring with a Python Notebook\n\nThe data...,NaN,NaN,markdown,19349
2,2,19349,import pandas as pd\nimport sqlite3\ncon = sql...,NaN,NaN,code,19349
3,3,19349,Let's explore the top 10 competitions by the n...,NaN,NaN,markdown,19349
4,4,19349,"print(pd.read_sql_query(""""""\nSELECT c.Competit...",NaN,NaN,code,19349
...,...,...,...,...,...,...,...
66701,66701,31804779,"data[""postcode""][data[""country""]==""Mexico""]",execute_result,30.0,code,31804779
66702,66702,31804779,24. Obtener la edad que más se repite en el Da...,NaN,NaN,markdown,31804779
66703,66703,31804779,"data[""age""].value_counts().sort_values(ascendi...",execute_result,31.0,code,31804779
66704,66704,31804779,25. Obtener la edad que menos se repite en el ...,NaN,NaN,markdown,31804779


In [16]:
SELECTED_WITH_MD_DF_PATH = (
    "/home/haotian/r/nb2p/nb2p/jupyter/24-resplit/data/nb2ptest_full_cells.csv"
)

In [ ]:
# selected_with_md_df.to_csv(SELECTED_WITH_MD_DF_PATH)

## Create IPython notebook Files

In [18]:
import json
from copy import deepcopy
# from pyminifier.minification import remove_comments_and_docstrings


class Cells2Nb:
    def __init__(self, blank_notebook_path="blank.ipynb"):
        self.blank_cell = {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {
                "collapsed": True,
                "editable": False,
                "jupyter": {"outputs_hidden": False},
            },
            "outputs": [],
            "source": [""],
        }

        with open(blank_notebook_path) as json_file:
            self.blank = json.load(json_file)

    def convert(self, cells_data, path="test.ipynb"):
        notebook = self.blank
        notebook["cells"] = [
            self.__generate_cell(cell_data) for cell_data in cells_data
        ]

        with open(path, "w") as json_file:
            json.dump(notebook, json_file)

    def __generate_cell(self, cell_data):
        cell = deepcopy(self.blank_cell)
        cell["cell_type"] = cell_data[0]
        cell["source"][0] = (
            cell_data[1]
            # remove_comments_and_docstrings(cell_data[1])
            # if cell_data[0] == "code"
            # else cell_data[1]
        )

        return cell

In [19]:
import os

import pandas as pd


def prepare_experiment(data_path="../24-resplit/data", path="../test"):
    # orig = pd.read_csv(f"{data_path}/orig.csv")
    # complete = pd.read_csv(f"{data_path}/complete.csv")

    cells_df = pd.read_csv(data_path)

    nbconverter = Cells2Nb()
    for num, nb_id in enumerate(nb_ids):
        if not os.path.exists(f"{path}/{nb_id}"):
            os.makedirs(f"{path}/{nb_id}")

        test = cells_df[cells_df["notebook_id"] == nb_id].to_dict(orient="records")
        test = [(c["cell_type"], c["source"]) for c in test]
        nbconverter.convert(test, path=f"{path}/{nb_id}/notebook.ipynb")

    return None  # random_mapping

In [20]:
prepare_experiment(SELECTED_WITH_MD_DF_PATH)